# 01 — Match Rate Predikat Provinsi

**Tahap 4 ceklis: Audit ulang match rate** · Protokol DSIC-2906-1 bagian 7 (T-01), 12 (K-01), 15 (K-04)

**Pertanyaan:** apakah predikat wilayah tingkat **provinsi** mengisi rezim match rate menengah **2,5–10%** yang kosong pada audit awal (temuan T-01)?

**Definisi** (protokol bagian 5):

$$\text{match\_rate}(f) = \frac{\text{jumlah titik yang memenuhi predikat } f}{\text{jumlah seluruh titik terindeks}}$$

**Data:** `data/processed/corpus.parquet` (64.354 titik, snapshot `snap-4bdc85481fa2`). Kolom `province` berbentuk **daftar**, karena satu berita bisa tertaut ke atraksi di beberapa provinsi.

> Catatan: kolom provinsi yang dipakai adalah hasil perapian (`province_source`), sambil menunggu keputusan dosen soal K-01.

## Langkah 1–2: Membaca korpus

Korpus dibaca dari file Parquet hasil `src.corpus.build_corpus`. Jumlah baris harus **64.354**, sama dengan audit protokol.

In [2]:
# Membaca korpus hasil konstruksi (protokol bagian 16-17)
import pandas as pd

corpus = pd.read_parquet("../data/processed/corpus.parquet")
print(len(corpus))
corpus.head()

64354


,doc_id,chunk_id,text,source_type,source_id,entity_id,category,maturity_level,kabupaten,province,...,publisher_domain,publisher_id,story_lineage_id,published_at,fetched_at,snapshot_id,source_file,row_index,adwi_award,n_linked_attractions
0,atr:alam:21509:wisata_bunga_rafflesia,atr:alam:21509:wisata_bunga_rafflesia#0,Wisata Bunga Rafflesia Wisata alam. Data Sunga...,jadesta_atraksi,https://jadesta.kemenpar.go.id/atraksi/21509,[alam:21509:wisata_bunga_rafflesia],[alam],NaN,[Kabupaten Agam],[Sumatera Barat],...,NaN,NaN,NaN,NaT,NaN,snap-4bdc85481fa2,wisata_alam.csv,0.0,NaN,NaN
1,atr:alam:21498:air_terjun_talago_indah,atr:alam:21498:air_terjun_talago_indah#0,Air Terjun Talago Indah Wisata alam. Nagari Pa...,jadesta_atraksi,https://jadesta.kemenpar.go.id/atraksi/21498,[alam:21498:air_terjun_talago_indah],[alam],NaN,[Kabupaten Agam],[Sumatera Barat],...,NaN,NaN,NaN,NaT,NaN,snap-4bdc85481fa2,wisata_alam.csv,1.0,NaN,NaN
2,atr:alam:20961:taman_wisata_mukomuko,atr:alam:20961:taman_wisata_mukomuko#0,Taman Wisata Muko-muko Wisata alam. Jorong Muk...,jadesta_atraksi,https://jadesta.kemenpar.go.id/atraksi/20961,[alam:20961:taman_wisata_mukomuko],[alam],NaN,[Kabupaten Agam],[Sumatera Barat],...,NaN,NaN,NaN,NaT,NaN,snap-4bdc85481fa2,wisata_alam.csv,2.0,NaN,NaN
3,atr:alam:2664:wisata_alam_kayu_gadang,atr:alam:2664:wisata_alam_kayu_gadang#0,Wisata Alam Kayu Gadang Wisata alam.,jadesta_atraksi,https://jadesta.kemenpar.go.id/atraksi/2664,[alam:2664:wisata_alam_kayu_gadang],[alam],NaN,[Kabupaten Agam],[Sumatera Barat],...,NaN,NaN,NaN,NaT,NaN,snap-4bdc85481fa2,wisata_alam.csv,3.0,NaN,NaN
4,atr:alam:20860:paralayang_4,atr:alam:20860:paralayang_4#0,Paralayang Wisata alam. Desa Wisata Bayua Fasi...,jadesta_atraksi,https://jadesta.kemenpar.go.id/atraksi/20860,[alam:20860:paralayang_4],[alam],NaN,[Kabupaten Agam],[Sumatera Barat],...,NaN,NaN,NaN,NaT,NaN,snap-4bdc85481fa2,wisata_alam.csv,4.0,NaN,NaN


Isi kolom `province` diapit kurung siku, artinya bertipe **daftar (list)**. Untuk dokumen JADESTA isinya selalu satu provinsi.

In [3]:
corpus[["source_type", "province"]].head(10)

,source_type,province
0,jadesta_atraksi,[Sumatera Barat]
1,jadesta_atraksi,[Sumatera Barat]
2,jadesta_atraksi,[Sumatera Barat]
3,jadesta_atraksi,[Sumatera Barat]
4,jadesta_atraksi,[Sumatera Barat]
5,jadesta_atraksi,[Sumatera Barat]
6,jadesta_atraksi,[Sumatera Barat]
7,jadesta_atraksi,[Sumatera Barat]
8,jadesta_atraksi,[Sumatera Barat]
9,jadesta_atraksi,[Sumatera Barat]


## Langkah 2b: Melihat dokumen berita

Berita bisa tertaut ke **lebih dari satu provinsi**. Contoh: *"Trenggalek Kembangkan Wisata Selam di Pantai Mutiara"* tercatat di DKI Jakarta dan Jawa Timur, karena di Kepulauan Seribu juga ada atraksi bernama "Pantai Mutiara" (pencocokan berdasarkan nama). Baris 21466 dan 21467 juga contoh **sindikasi**, yaitu berita yang sama dari dua penerbit.

In [4]:
# Subset korpus yang hanya berisi berita, dan 10 berita pertamanya
berita = corpus[corpus["source_type"] == "news"]
berita[["text", "province"]].head(10)

,text,province
21461,"Festival Sangiang Api Dibidik Masuk KEN, Poten...","[Jawa Tengah, Nusa Tenggara Barat, Nusa Tengga..."
21462,"Wisata Kota Batu Terbaru dan Populer, Ada Tama...",[Jawa Timur]
21463,POLSUB Perkuat Wisata Leuwi Lawang: Sinergi Ke...,"[Jawa Barat, Kepulauan Riau]"
21464,Festival Sangiang Api 2026: Angkat Sejarah dan...,[Nusa Tenggara Barat]
21465,"Pantai Indah Kemangi Kendal, Dari Wisata Panta...",[Jawa Tengah]
21466,Trenggalek Kembangkan Wisata Selam di Pantai M...,"[DKI Jakarta, Jawa Timur]"
21467,Trenggalek kembangkan wisata selam Pantai Mutiara,"[DKI Jakarta, Jawa Timur]"
21468,"Wisata Alam Citalutug, Wisata Alam Kreatif Kek...",[Jawa Barat]
21469,DLAS Serang Purbalingga Siapkan Paket Liburan ...,[Jawa Tengah]
21470,Wisata Tebing Apparalang Bulukumba Ternyata Ta...,[Sulawesi Selatan]


### Sebaran jumlah provinsi per berita

Menghitung panjang daftar `province` untuk setiap berita, untuk mengetahui seberapa banyak berita yang metadata wilayahnya **ambigu**.

In [ ]:
# Menghitung berapa provinsi yang dimiliki setiap berita (panjang daftar)
jumlah_provinsi = berita["province"].str.len()

# Menyaring berita yang provinsinya lebih dari satu, lalu hitung jumlahnya
multi_provinsi = berita[jumlah_provinsi > 1]
print("Berita dengan >1 provinsi:", len(multi_provinsi))
print("Persentase:", round(len(multi_provinsi) / len(berita) * 100, 1), "%")

# Sebaran: berapa berita punya 1, 2, 3, ... provinsi
print(jumlah_provinsi.value_counts().sort_index())

Berita dengan >1 provinsi: 8923
Persentase: 20.8 %
province
1     33970
2      5059
3      1686
4       902
5       455
6       242
7       229
8       134
9        69
10       58
11       26
12       21
13       14
14        9
15        6
16        2
17        2
18        5
21        2
23        1
24        1
Name: count, dtype: int64


### Berita yang tertaut ke ≥ 20 provinsi

Judul-judulnya memuat kata umum yang juga menjadi **nama atraksi** di banyak daerah ("Camping Ground", "Puncak", "Danau"), sehingga satu berita tertaut ke atraksi bernama sama di seluruh Indonesia. Ini bukti bahwa metadata wilayah berita dari pencocokan nama **tidak andal** untuk berita multi-provinsi.

In [6]:
berita[jumlah_provinsi >= 20][["text", "province"]]

,text,province
22445,"Sinabung Waspada, Wisata Danau Lau Kawar dan C...","[Aceh, Bali, Bengkulu, DI Yogyakarta, Gorontal..."
24663,6 Rekomendasi Camping Ground di Surabaya dan S...,"[Aceh, Bali, Bengkulu, DI Yogyakarta, Gorontal..."
26259,Kemenhut Perkuat Pembinaan Saka Wanabakti di N...,"[Aceh, Bali, Bengkulu, DI Yogyakarta, DKI Jaka..."
27031,Video: JALAN YUK! Hidden Gem di Puncak! Ada Re...,"[Bali, Bengkulu, DI Yogyakarta, DKI Jakarta, G..."


## Langkah 3: Memecah daftar provinsi dengan `explode()`

Setiap pasangan (dokumen, provinsi) menjadi satu baris. Berita dengan 2 provinsi menjadi 2 baris dengan `doc_id` yang sama. Pengecekan: jumlah baris sesudah explode harus sama dengan total isi semua daftar.

In [7]:
# Memecah kolom province: satu baris per (dokumen, provinsi)
corpus_explode = corpus.explode("province")

print("Baris sebelum explode:", len(corpus))
print("Baris sesudah explode:", len(corpus_explode))

# Cek: jumlah baris sesudah explode = total isi semua daftar provinsi
print("Total isi daftar     :", corpus["province"].str.len().sum())

Baris sebelum explode: 64354
Baris sesudah explode: 82999
Total isi daftar     : 82999


## Langkah 4: Match rate per provinsi

Jumlah titik per provinsi dibagi **N = 64.354** (seluruh titik terindeks), **bukan** 82.999. Explode hanya cara menghitung, tidak menambah titik di indeks.

In [8]:
# Jumlah seluruh dokumen asli (64.354), dipakai sebagai pembagi match rate
N = len(corpus)

# Jumlah titik untuk setiap provinsi, dan persentasenya terhadap seluruh titik
jumlah_per_provinsi = corpus_explode["province"].value_counts()
match_rate_provinsi = jumlah_per_provinsi / N * 100

# Tabel gabungan berisi jumlah titik dan match rate per provinsi
tabel_provinsi = pd.DataFrame({
    "jumlah_titik": jumlah_per_provinsi,
    "match_rate_%": match_rate_provinsi.round(3),
})
tabel_provinsi

,jumlah_titik,match_rate_%
province,,
Jawa Timur,8649,13.440
Sulawesi Selatan,7341,11.407
Jawa Tengah,6569,10.208
Jawa Barat,6420,9.976
Sumatera Barat,5875,9.129
DI Yogyakarta,5232,8.130
Nusa Tenggara Barat,4049,6.292
Sumatera Utara,3277,5.092
Sulawesi Utara,2412,3.748


## Langkah 5: Pengelompokan ke bin match rate

Batas bin mengikuti usulan protokol bagian 15 (K-04): **B1** < 0,1% · **B2** 0,1–0,5% · **B3** 0,5–2,5% · **B4** 2,5–10% · **B5** > 10%.

Sekaligus mengecek syarat kecukupan protokol bagian 18: setiap predikat harus punya **≥ 100 titik** (10 × k, dengan k = 10).

In [9]:
# Batas dan nama bin match rate sesuai protokol bagian 15 (dalam persen)
batas = [0, 0.1, 0.5, 2.5, 10, 100]
label = ["B1 (<0,1%)", "B2 (0,1-0,5%)", "B3 (0,5-2,5%)", "B4 (2,5-10%)", "B5 (>10%)"]

# Kolom bin untuk setiap provinsi berdasarkan match rate-nya
tabel_provinsi["bin"] = pd.cut(tabel_provinsi["match_rate_%"], bins=batas, labels=label)

# Jumlah provinsi di setiap bin
print(tabel_provinsi["bin"].value_counts().sort_index())

# Provinsi yang tidak memenuhi syarat kecukupan 10 x k (kurang dari 100 titik)
print("\nProvinsi < 100 titik:")
print(tabel_provinsi[tabel_provinsi["jumlah_titik"] < 100])

bin
B1 (<0,1%)        2
B2 (0,1-0,5%)     1
B3 (0,5-2,5%)    16
B4 (2,5-10%)     16
B5 (>10%)         3
Name: count, dtype: int64

Provinsi < 100 titik:
               jumlah_titik  match_rate_%         bin
province                                             
Papua Selatan             1         0.002  B1 (<0,1%)
Papua Tengah              1         0.002  B1 (<0,1%)


## Langkah 6: Match rate provinsi untuk dokumen JADESTA saja

Perhitungan yang sama, tetapi tanpa berita (N = 21.461). Tujuannya melihat seberapa besar berita, terutama yang ambigu, memengaruhi match rate provinsi.

In [10]:
# Subset korpus yang hanya berisi dokumen JADESTA (atraksi dan desa wisata)
jadesta = corpus[corpus["source_type"] != "news"]
N_jadesta = len(jadesta)

# Jumlah titik, match rate, dan bin per provinsi untuk dokumen JADESTA
jumlah_jd = jadesta.explode("province")["province"].value_counts()
tabel_jd = pd.DataFrame({
    "jumlah_titik": jumlah_jd,
    "match_rate_%": (jumlah_jd / N_jadesta * 100).round(3),
})
tabel_jd["bin"] = pd.cut(tabel_jd["match_rate_%"], bins=batas, labels=label)

print("Total dokumen JADESTA:", N_jadesta)
print(tabel_jd["bin"].value_counts().sort_index())

Total dokumen JADESTA: 21461
bin
B1 (<0,1%)        2
B2 (0,1-0,5%)     2
B3 (0,5-2,5%)    23
B4 (2,5-10%)     10
B5 (>10%)         1
Name: count, dtype: int64


## Langkah 7: Perbandingan semua dokumen vs JADESTA saja

Match rate kedua versi ditampilkan berdampingan, diurutkan dari yang tertinggi.

In [11]:
# Match rate provinsi dari seluruh dokumen dan dari JADESTA saja, dalam satu tabel
banding = tabel_provinsi[["match_rate_%"]].join(
    tabel_jd[["match_rate_%"]], lsuffix="_semua", rsuffix="_jadesta"
)
banding.sort_values("match_rate_%_semua", ascending=False)

,match_rate_%_semua,match_rate_%_jadesta
province,,
Jawa Timur,13.440,10.801
Sulawesi Selatan,11.407,9.012
Jawa Tengah,10.208,8.411
Jawa Barat,9.976,7.139
Sumatera Barat,9.129,9.021
DI Yogyakarta,8.130,5.526
Nusa Tenggara Barat,6.292,4.250
Sumatera Utara,5.092,4.389
Sulawesi Utara,3.748,2.348


## Ringkasan hasil

| | Semua dokumen (N = 64.354) | JADESTA saja (N = 21.461) |
|---|---:|---:|
| B1 (< 0,1%) | 2 | 2 |
| B2 (0,1–0,5%) | 1 | 2 |
| B3 (0,5–2,5%) | 16 | 23 |
| **B4 (2,5–10%)** | **16** | **10** |
| B5 (> 10%) | 3 | 1 |

- **Rezim menengah (B4) terisi**: 16 provinsi pada semua dokumen, 10 provinsi pada JADESTA saja. Pada audit awal (tingkat kabupaten) rezim ini hampir kosong.
- **B5 (> 10%)**: Jawa Timur (13,44%), Sulawesi Selatan (11,41%), Jawa Tengah (10,21%).
- **Tidak memenuhi syarat ≥ 100 titik**: Papua Selatan dan Papua Tengah, masing-masing hanya **1 titik**. Keduanya tidak bisa dipakai sebagai predikat kueri.
- **Berita menaikkan match rate** hampir semua provinsi (contoh: Jawa Timur 10,80% → 13,44%; DKI Jakarta 1,31% → 2,59%). Sebagian kenaikan ini berasal dari berita yang tertaut ke banyak provinsi karena pencocokan nama.
- 8.923 berita (20,8%) tertaut ke > 1 provinsi; 4 berita tertaut ke ≥ 20 provinsi.

## Kesimpulan dan implikasi

## Kesimpulan

1. **Predikat provinsi mengisi rezim match rate menengah.** Pada seluruh korpus,
   16 provinsi berada di B4 (2,5–10%); pada dokumen JADESTA saja, 10 provinsi.
   Rezim ini hampir kosong pada audit awal yang hanya memakai predikat kabupaten
   (temuan T-01). Dengan predikat provinsi, kurva recall–QPS berpeluang
   menunjukkan titik peralihan antara rezim selektif dan rezim lebar.

2. **36 dari 38 provinsi memenuhi syarat kecukupan (≥ 100 titik).** Papua Selatan
   dan Papua Tengah masing-masing hanya memiliki 1 titik, sehingga tidak dapat
   dipakai sebagai predikat kueri.

3. **Provinsi juga mengisi rezim lebar.** Jawa Timur (13,44%), Sulawesi Selatan
   (11,41%), dan Jawa Tengah (10,21%) berada di B5. Dengan demikian B5 tidak hanya
   berisi predikat kategori, tipe sumber, dan temporal.

4. **Metadata wilayah berita tidak sepenuhnya andal.** Sebanyak 8.923 berita
   (20,8%) tertaut ke lebih dari satu provinsi karena pencocokan berita–atraksi
   berbasis nama. Contoh: berita tentang Pantai Mutiara di Trenggalek ikut tertaut
   ke DKI Jakarta, dan berita bertema "Camping Ground" tertaut ke lebih dari 20
   provinsi.

## Implikasi

1. **K-01:** predikat provinsi layak dipertahankan karena menjawab T-01. Yang
   perlu diputuskan saat bimbingan adalah sumber kolomnya: kolom provinsi hasil
   perapian, atau penurunan ulang melalui tabel pemetaan kabupaten → provinsi
   (rekomendasi protokol).

2. **Penempatan provinsi ke bin sensitif terhadap perlakuan berita ambigu.**
   Aturan "cocok jika salah satu provinsi cocok" menaikkan jumlah titik provinsi.
   Karena itu aturan perlakuan berita ambigu harus ditetapkan **sebelum** batas bin
   dan himpunan kueri dibekukan (K-04, Tahap 5).

3. **Papua Selatan dan Papua Tengah dikeluarkan** dari populasi predikat provinsi.

4. **Ancaman validitas:** kualitas metadata wilayah berita memengaruhi himpunan
   titik eligible dan ground truth. Hal ini perlu dinyatakan pada bagian
   Threats to Validity, sejalan dengan README bagian 38 (Metadata Quality).

## Catatan metodologis

Perbandingan "semua dokumen" vs "JADESTA saja" memakai pembagi yang berbeda
(64.354 vs 21.461), sehingga yang dibandingkan adalah komposisi korpus, bukan
besar inflasi akibat berita ambigu. Untuk mengukur inflasi secara langsung,
match rate perlu dihitung ulang dengan pembagi yang sama (N = 64.354) tetapi
berita ambigu tidak dihitung pada predikat provinsi.